In [1]:
import numpy as np
import rebound
import matplotlib.pyplot as plt
import os
import pandas as pd
from datetime import datetime

os.chdir('../src/')


import sbdynt as sbd

# Default Proper Element Run

Default properties for computing proper elements for TNOs is contained within the same ``run_tno`` function used to initialize and run the machine learning algorithm for TNO resonance occupation.
A similar function, ``run_asteroid`` can be used to similarly run and compute proper elements for asteroids. We demonstrate these functions below for TNO (15760) Albion and the dwarf planet and asteroid (1) Ceres. 

To compute proper elements as well, from the ``run_tno`` function, user should set the boolean variable ``run_proper = True``, which will compute the proper elements and save the results to the ``tno_result`` output. 

The bulk of the time required by these functions is actually spent integrating the simulation, or if the simulation is already integrated, in reading the orbital elements from the Simulation Archive binary files. The actual computation of the proper elements is comparatively fast.

In this TNO example, we will still run the machine learning algorithm by including ``run_ML = True``, as this produces a more complex Simulation Archive with varying time resolution. This demonstrates how these complex archives are automatically handled while computing the proper elements from the simulation. We also include print statements to demonstrate and benchmark how long the default runs may take to fully integrate. 

In [2]:
begin = datetime.now()
tno_result = sbd.run_tno(des='Albion', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, 
                                     logfile=False,deletefile=True, run_ML = True, run_proper=True)
print(datetime.now()-begin)


begin = datetime.now()
ast_result = sbd.run_ast(des='Ceres', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None,
                                     logfile=False,deletefile=True)
print(datetime.now()-begin)

Running TNO ML
Running TNO integration for PE/Stability Indicators
Reading TNO integration
Running TNO PE
0:23:17.020052
Running Asteroid integration for Proper Elements
Reading Asteroid integration
0:22:30.765770


Functions are also provided to use an existing Simulation archive for a TNO or asteroid to compute the proper elements from that archive file, which bypasses the need to reintegrate the entire small body.

An archive file that is not computed using SBDynT may be used, but user must make sure that the archive file has the appropriate planets for the object with the correct hashes as identifiers, and must also make sure that the proper ``object_type`` is specified. However, we recommend typically implementing these functions only with archive files produced using SBDynT code, as formatting differences may occur. 

Since we have already integrated Simulation archives above for TNO ``(15760) Albion`` and asteroid ``(1) Ceres``, we may use these functions to more quickly compute the proper elements for these objects.

In [3]:
begin = datetime.now()
tno_result = sbd.run_existing_tno(des='Albion', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, 
                                     logfile=False, run_ML = False, run_proper=True, output_arrays = True)
print(datetime.now()-begin)


begin = datetime.now()
ast_result = sbd.run_existing_ast(des='Ceres', clones=1, datadir='../example-notebooks/example_sims/',archivefile=None, object_type='asteroid',
                                     logfile=False, run_proper=True, output_arrays = True)
print(datetime.now()-begin)

Reading TNO integration for Proper Elements and/or Chaos
Running TNO Proper Elements
0:00:14.261486
Reading Asteroid integration
Running Asteroid Proper Elements
0:00:09.635968


We can see the results in an easy way by calling the ``print_results`` function within the proper_element class object inside of the result.

In [4]:
tno_result.proper_elements.print_results()
ast_result.proper_elements.print_results()

Albion  Proper Element Results from a 150 Myr integration with 5000 year outputs
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 43.9325 	 0.06938 	 2.18552 	 N/A    	 N/A
#Mean Elements: 	 43.92894 	 0.07058 	 2.83204 	 0.41122 	 -0.32043
#Proper Elements: 	 43.92894 	 0.0699 	 2.514  	 0.41819 	 -0.42563
#Proper Errors: 	 3.783e-05 	 7.844e-04 	 1.857e-02 	 6.249e-03 	 7.698e-04

Ceres  Proper Element Results from a 10 Myr integration with 500 year outputs
# 			 SMA(AU) 	 Ecc    	 Inc(deg) 	 g("/yr) 	 s("/yr)
#Osculating Elements: 	 2.74689 	 0.08057 	 10.60557 	 N/A    	 N/A
#Mean Elements: 	 2.76341 	 0.11808 	 9.71923 	 51.12225 	 -56.79386
#Proper Elements: 	 2.76341 	 0.11502 	 9.64198  	 54.288 	 -59.22766
#Proper Errors: 	 8.840e-06 	 1.072e-04 	 2.323e-04 	 1.041e-01 	 6.933e-02



# Proper Element Outputs

The proper element results are saved to a ``proper_element`` class object within the ``tno_result`` object. 
This class object contains a large number of relevant values and indicators related to the orbital evolution of the small body corresponding to the proper motion over time. These include...

```proper_elements```

```mean_elements```

```osculating_elements```

```proper_errors```

```planet_freqs```

```proper_windows```

### Proper Elements

The proper elements themselves are contained in a dictionary with the same name as the ``proper_element`` class. These can be contrasted with the mean elements, (which are simply the mean of the osculating element time array), and the initial osculating elements, (which represent the orbital elements at time ``t=0`` in the simulation).

We note that these mean elements are not be confused with the ``mean`` indicator, which is used to identify objects which experience chaotic motion or long-term periodic evolution such that the synethic proper element cannot be accurately computed for the small-body in the given integration. 
We will discuss the ``mean`` indicator, as well as other useful indicators, in more detail in the ``proper_elements_advanced`` notebook. 

These three outputs contain the semi-major axis, eccentricity, inclination, as well as the proper and mean precession rates, ``g`` and ``s``, in the case of the ``proper_elements`` and ``mean_elements`` variables. The ``osculating_elements`` variable instead reports the initial ``omega`` and ``Omega`` values at the ``t=0`` epoch.

The proper and mean ``g`` and ``s`` frequencies are reported in both revolutions/year (the units of measurement used by the filtering, the direct result from the filter) and arcseconds/year (the typically reported value for the proper precession rates).

In [5]:
print('Albion: ',tno_result.proper_elements.proper_elements)
print('\nCeres:', ast_result.proper_elements.proper_elements)


Albion:  {'a': 43.92893681520119, 'e': 0.06990455017952323, 'sinI': 0.043863578628668895, 'g(rev/yr)': 3.226798138000856e-07, 's(rev/yr)': -3.2841755739379213e-07, 'g("/yr)': 0.418193038684911, 's("/yr)': -0.42562915438235455, 'omega': array([1.43086162]), 'Omega': array([-0.94935382])}

Ceres: {'a': 2.763406515668239, 'e': 0.11501572461489104, 'sinI': 0.16749116448114962, 'g(rev/yr)': 4.188889189641937e-05, 's(rev/yr)': -4.570035823404694e-05, 'g("/yr)': 54.28800389775951, 's("/yr)': -59.227664271324834, 'omega': array([1.4786565]), 'Omega': array([1.35448985])}


In [6]:
print('Albion:', tno_result.proper_elements.mean_elements)
print('\nCeres:', ast_result.proper_elements.mean_elements)

Albion: {'a': 43.928936815201205, 'e': 0.07057534099719842, 'sinI': 0.04940822975845391, 'g(rev/yr)': 3.1730131879629915e-07, 's(rev/yr)': -2.472439983964198e-07, 'g("/yr)': 0.4112225091600037, 's("/yr)': -0.32042822192176007}

Ceres: {'a': 2.763406515668238, 'e': 0.11807830194545915, 'sinI': 0.16882018760934106, 'g(rev/yr)': 3.9446182755905544e-05, 's(rev/yr)': -4.382241912399143e-05, 'g("/yr)': 51.122252851653585, 's("/yr)': -56.7938551846929}


In [7]:
print('Albion:',tno_result.proper_elements.osculating_elements)
print('\nCeres:',ast_result.proper_elements.osculating_elements)

Albion: {'a': array([43.93250143]), 'e': array([0.06937744]), 'I': array([0.03814445]), 'omega': array([0.04935982]), 'Omega': array([-0.01023984])}

Ceres: {'a': array([2.74688581]), 'e': array([0.08056522]), 'I': array([0.18510206]), 'omega': array([1.23614904]), 'Omega': array([1.40037391])}


### Proper Element Uncertainties

The proper element uncertainties are contained in the ``proper_errors`` dictionary.

In [8]:
print('Albion:', tno_result.proper_elements.proper_errors)
print('\nCeres:',ast_result.proper_elements.proper_errors)

Albion: {'RMS_a': 3.783184569759844e-05, 'RMS_e': 0.0007843964754131926, 'RMS_sinI': 0.0003241835307268976, 'RMS_g(rev/yr)': 4.822047313643417e-09, 'RMS_s(rev/yr)': 5.939998307475412e-10, 'RMS_g("/yr)': 0.006249373318481868, 'RMS_s("/yr)': 0.0007698237806488134}

Ceres: {'RMS_a': 8.84006673691857e-06, 'RMS_e': 0.00010724019405388487, 'RMS_sinI': 4.054596418167904e-06, 'RMS_g(rev/yr)': 8.029798735757562e-08, 'RMS_s(rev/yr)': 5.3498381531960903e-08, 'RMS_g("/yr)': 0.104066191615418, 'RMS_s("/yr)': 0.06933390246542133}


### Planetary Frequencies

Users interested in the identified planetary secular frequencies can retrieve these from the ``planets`` dictionary. These are only reported in the units of rev/yr. Users can retrieve the "/yr units by multipying these results by 1296000.

In [9]:
print('Albion:', tno_result.proper_elements.planet_freqs)
print('\nCeres:', ast_result.proper_elements.planet_freqs)

Albion: {'g5': 3.271591161504451e-06, 'g6': 2.1785958248492846e-05, 'g7': 2.3791363888925526e-06, 'g8': 5.214146032461065e-07, 's6': -2.0328348498629446e-05, 's7': -2.3113078407711607e-06, 's8': -5.348589013533798e-07}

Ceres: {'g5': 3.2999222666243006e-06, 'g6': 2.1799972548635788e-05, 'g7': 2.3998158233617146e-06, 'g8': 4.999358620998731e-07, 's6': -2.03038604885051e-05, 's7': -2.300005105070441e-06, 's8': -5.060638049212581e-07, 'g2': 5.513626202629028e-06, 'g3': 1.3609004019681211e-05, 'g4': 1.3116802920676387e-05, 's3': -1.4456233703194622e-05, 's4': -1.3697664700069688e-05, 's2': -4.794454843482448e-06}


### Window Proper Elements

Users may also access the proper element computed for each of the windows. This could be useful in cases of objects which experience chaotic or scattering motion, if the users wishes to see the proper element before such events. 

In [10]:
print('Albion:', tno_result.proper_elements.proper_windows)
print('\nCeres:', ast_result.proper_elements.proper_windows)

Albion: {'a_win': array([43.92888641, 43.92889849, 43.92891188, 43.92893956, 43.92898699]), 'e_win': array([0.06970752, 0.06945736, 0.06947274, 0.06835143, 0.06941574]), 'sinI_win': array([0.04391134, 0.0443103 , 0.04419152, 0.04372167, 0.04430624]), 'g_win': array([3.24883896e-07, 3.27927361e-07, 3.28444527e-07, 3.28520130e-07,
       3.26744879e-07]), 's_win': array([-3.27140980e-07, -3.28364929e-07, -3.28545542e-07, -3.28622423e-07,
       -3.28146614e-07])}

Ceres: {'a_win': array([2.76341246, 2.76341751, 2.76341865, 2.76341305, 2.76339986]), 'e_win': array([0.11513958, 0.11518084, 0.11513251, 0.11501898, 0.1149804 ]), 'sinI_win': array([0.16749915, 0.1674936 , 0.16749128, 0.16748948, 0.16748805]), 'g_win': array([4.18081801e-05, 4.18085830e-05, 4.18088407e-05, 4.18088639e-05,
       4.18085038e-05]), 's_win': array([-4.57706028e-05, -4.57616194e-05, -4.57481696e-05, -4.57419443e-05,
       -4.57404559e-05])}


More advanced outputs are further discussed in the ``proper_elements_advanced.ipynb`` file, which include flags related to chaos, the amplitude of secular terms, as well as some indicators which may be hekpful for identifying secualr resonance occupation. 